In [29]:
#setup Java environment for Spark
# This is for Windows OS
# import os

# os.environ["JAVA_HOME"] = r"C:\\Program Files\\Java\\jdk1.8.0_202"
# os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# create local spark session
from pyspark.sql import SparkSession
from pyspark .sql.functions import col, date_format
from pyspark.sql.types import TimestampType

import pandas as pd

spark = SparkSession.builder.master("local[*]").appName("postgres_upload_ny_taxi").getOrCreate()

# read the parquet data from /data folder into a dataframe
df = spark.read.parquet("data/ny_taxi_data_yellow/yellow_tripdata_2021-01.parquet")


# Convert timestamp columns to string first
timestamp_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]

for c in timestamp_cols:
    df = df.withColumn(c, date_format(col(c), "yyyy-MM-dd HH:mm:ss"))

In [55]:
# volume of data
df.count()

1369769

In [30]:
# data showcase/ schema showcase
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [37]:
# convert to pandas
pdf = df.toPandas()

for c in timestamp_cols:
    pdf[c] = pd.to_datetime(pdf[c])

In [38]:
from sqlalchemy import create_engine
# Example credentials
username = "root"
password = "root"
host = "localhost"
port = 5432
database = "ny_taxi"

# Create the engin
engine = create_engine(f"postgresql://{username}:{password}@{host}:{port}/{database}")

In [39]:
engine.connect()

In [40]:
create_statement_postgres=pd.io.sql.get_schema(pdf,name="yellow_tax_data",con=engine)

In [41]:
print(create_statement_postgres)


CREATE TABLE yellow_tax_data (
	"VendorID" BIGINT, 
	tpep_pickup_datetime TIMESTAMP WITHOUT TIME ZONE, 
	tpep_dropoff_datetime TIMESTAMP WITHOUT TIME ZONE, 
	passenger_count FLOAT(53), 
	trip_distance FLOAT(53), 
	"RatecodeID" FLOAT(53), 
	store_and_fwd_flag TEXT, 
	"PULocationID" BIGINT, 
	"DOLocationID" BIGINT, 
	payment_type BIGINT, 
	fare_amount FLOAT(53), 
	extra FLOAT(53), 
	mta_tax FLOAT(53), 
	tip_amount FLOAT(53), 
	tolls_amount FLOAT(53), 
	improvement_surcharge FLOAT(53), 
	total_amount FLOAT(53), 
	congestion_surcharge FLOAT(53), 
	airport_fee FLOAT(53)
)




In [43]:
# creating and inserting data in yellow_taxi_data 

# create yellow_taxi_data 
pdf.head(0).to_sql(name="yellow_taxi_date", con=engine, if_exists="append")

0

In [59]:
# insert chunks of 10000
from time import time

chunksize = 10000
for i in range(0, len(pdf), chunksize):
    t_start = time()
    pdf.iloc[i:i+chunksize].to_sql(name="yellow_taxi_date", con=engine, if_exists="append")
    t_end = time()
    print(f"Inserted data successfully in {t_end - t_start:.3f}s")
    

Inserted data successfully in 2.042s
Inserted data successfully in 1.616s
Inserted data successfully in 1.377s
Inserted data successfully in 2.804s
Inserted data successfully in 1.504s
Inserted data successfully in 1.330s
Inserted data successfully in 1.553s
Inserted data successfully in 1.284s
Inserted data successfully in 1.507s
Inserted data successfully in 1.702s
Inserted data successfully in 1.354s
Inserted data successfully in 1.442s
Inserted data successfully in 1.445s
Inserted data successfully in 1.550s
Inserted data successfully in 1.347s
Inserted data successfully in 1.954s
Inserted data successfully in 1.457s
Inserted data successfully in 1.300s
Inserted data successfully in 1.355s
Inserted data successfully in 1.455s
Inserted data successfully in 1.423s
Inserted data successfully in 1.543s
Inserted data successfully in 1.316s
Inserted data successfully in 1.450s
Inserted data successfully in 1.373s
Inserted data successfully in 1.494s
Inserted data successfully in 1.585s
I